In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)
df = pd.read_parquet("../../Database/data/eda_banking.parquet")
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,11983969.0,1,1,1,10134888.0,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,8380786.0,1,0,1,11254258.0,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,15966080.0,3,1,0,11393157.0,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,11983969.0,2,0,0,9382663.0,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,12551082.0,1,1,1,7908410.0,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,11983969.0,2,1,0,9627064.0,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,5736961.0,1,1,1,10169977.0,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,11983969.0,1,0,1,4208558.0,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,7507531.0,2,1,0,9288852.0,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Create postgresql configure connection

In [41]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

DIR_PATH = os.getcwd().partition("ml-engineer")[0]
ENV_PATH = os.path.join(DIR_PATH, ".env")
load_dotenv(ENV_PATH)

# Get database credentials from environment variables
DB_USER = os.getenv("DB_USER")
DB_NAME = os.getenv("DB_NAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")

# Create engine to connect to PostgreSQL
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Write DataFrame to PostgreSQL
try:
    df.to_sql("banking_data", con=engine, if_exists="replace", index=False, schema="bank_pipeline")
    print("DataFrame successfully written to PostgreSQL.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame successfully written to PostgreSQL.


In [38]:
from sqlalchemy import text

# Query data from PostgreSQL using connection
with engine.connect() as connection:
    query_result = pd.read_sql(text("SELECT * FROM banking_data"), connection)
    print(query_result.head())

   CreditScore Geography  Gender  Age  Tenure     Balance  NumOfProducts  \
0          619    France  Female   42       2  11983969.0              1   
1          608     Spain  Female   41       1   8380786.0              1   
2          502    France  Female   42       8  15966080.0              3   
3          699    France  Female   39       1  11983969.0              2   
4          850     Spain  Female   43       2  12551082.0              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  Complain  \
0          1               1       10134888.0       1         1   
1          0               1       11254258.0       0         1   
2          1               0       11393157.0       1         1   
3          0               0        9382663.0       0         0   
4          1               1        7908410.0       0         0   

   Satisfaction Score Card Type  Point Earned  Fraud  RiskScore  \
0                   2   DIAMOND           464      0          1   
1     